## 1. Read data

### 1.1. The dataset of sentences

In [ ]:
import pandas as pd
sentence_df = pd.read_excel("/workspace/mijnidbcoachnlp/data/analysis_datasets/translated_sentence_data.xlsx")

In [ ]:
sentences = sentence_df["Clean_Sentence"].to_list()

### 1.2. Import list of stopwords

In [ ]:
### Importing the list of Dutch stopwords (note that there are customized dutch words in there)

with open('/workspace/mijnidbcoachnlp/data/analysis_datasets/stopwords.txt', 'r') as file:
    lines = [line.strip() for line in file.readlines()]

dutch_stopwords = lines

extra_list = [
    'maandag', 'dinsdag', 'woensdag', 'donderdag', 'vrijdag', 'zaterdag', 'zondag',
    'januari', 'februari', 'maart', 'april', 'mei', 'juni', 'juli', 'augustus', 'september', 'oktober', 'november', 'december',
    'jan', 'feb', 'mrt', 'apr', 'mei', 'jun', 'jul', 'aug', 'sep', 'okt', 'nov', 'dec', "jl"
]

dutch_stopwords.extend(extra_list)

## 2. Download the embedding model BERTje

### 2.1 Download BERTje

In [ ]:
from transformers.pipelines import pipeline
from transformers import AutoTokenizer, AutoModel, TFAutoModel
embedding_model = pipeline("feature-extraction", model="GroNLP/bert-base-dutch-cased")

### 2.2 Pre-calculate and save sentence embeddings (skip if there are saved embeddings and go to 2.3)

In [ ]:
# pre-calculate the embeddings of the dutch sentences
import numpy as np

embeddings_nl = embedding_model(sentences, truncation=True, padding=True)
sentence_embeddings_nl = np.array([np.mean(embedding, axis=1).flatten() for embedding in embeddings_nl])

In [ ]:
from tqdm import tqdm 
# Initialize an empty list to store embeddings
sentence_embeddings_nl = []

# Use tqdm to track progress
for sentence in tqdm(sentences, desc="Processing Embeddings", unit="sentence"):
    # Compute embedding for the sentence
    embedding = embedding_model(sentence)
    # Average token embeddings to create a sentence-level embedding
    sentence_embedding = np.mean(embedding[0], axis=0)
    # Append to the list
    sentence_embeddings_nl.append(sentence_embedding)

# Convert the list to a NumPy array
sentence_embeddings_nl = np.array(sentence_embeddings_nl)

# Check the shape of the resulting embeddings
print(f"Shape of sentence embeddings: {sentence_embeddings_nl.shape}")

In [ ]:
# save the sentence embeddings
import numpy as np
np.save('/workspace/mijnidbcoachnlp/data/embeddings/sentence_embeddings_bertje_nl_new.npy', sentence_embeddings_nl)

### 2.3 Load saved embeddings

In [ ]:
# load the save model
import numpy as np

loaded_embeddings_nl = np.load('/workspace/mijnidbcoachnlp/data/embeddings/sentence_embeddings_bertje_nl.npy')

In [ ]:
embeddings=loaded_embeddings_nl

## 3. Preparation for Fine-Tuning Hyperparameters

### 3.1. Generate a list of hyperparameter combinations

In [ ]:
import itertools

# Generating a list of hyperparameter combinations
# UMAP hyperparameters
n_neighbors_values = [5, 10, 15, 20, 25, 30] # we are aiming for strict clusters so the n_neighbors range is relatively small
n_components_values = [2, 3, 4, 5]

# HDBSCAN hyperparameters
min_cluster_size_values = [10, 15, 20, 30]
min_samples_values = [5, 10, 15, 20, 30]

combinations = list(itertools.product(n_neighbors_values, n_components_values, min_cluster_size_values, min_samples_values))

# set min_samples < min_cluster_size
filtered_combinations = [
    combination for combination in combinations if combination[3] <= combination[2] 
]

# print the total number of combinations
len(filtered_combinations)

### 3.2 Function to calculate intra-topic similarity

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# function to calculate intra-topic cosine similarity
def calculate_intra_topic_cosine_similarity(embeddings, doc_topics):
    topic_similarities = []
    
    for topic in set(doc_topics):
        if topic == -1:  # Skip the "outlier" topic
            continue
        
        # Get embeddings of documents in the current topic
        topic_embeddings = embeddings[np.array(doc_topics) == topic]
        
        if len(topic_embeddings) < 2:
            continue  # Skip topics with a single document
        
        # Calculate cosine similarities and average them
        cosine_sim = cosine_similarity(topic_embeddings)
        avg_cosine_sim = cosine_sim[np.triu_indices_from(cosine_sim, k=1)].mean()
        
        topic_similarities.append(avg_cosine_sim)

    return topic_similarities

### 3.3. Function to calculate silhouette scores

In [ ]:
# function to calculate silhouette scores if needed (we don't use the silhouette scores for now because the clusters are probably overlapping a lot so inter-topic distance is not so informative)
from sklearn.metrics import silhouette_samples
import numpy as np

def calculate_intra_topic_silhouette_score(embeddings, topics):

    # Filter out documents assigned to the outlier topic (-1)
    mask = topics != -1
    filtered_embeddings = embeddings[mask]
    filtered_topics = topics[mask]

    # Compute silhouette scores
    silhouette_scores = silhouette_samples(filtered_embeddings, filtered_topics, metric="cosine")
    
    # Calculate average silhouette score for each topic
    unique_topics = np.unique(filtered_topics)
    avg_silhouette_per_topic = {
        topic: np.mean(silhouette_scores[filtered_topics == topic]) for topic in unique_topics
    }

    # Compute the overall average silhouette score across all topics
    avg_silhouette_score = np.mean(list(avg_silhouette_per_topic.values()))

    return avg_silhouette_score


### 3.4 Function to calculate coherence scores

In [ ]:
# create the dictionary of documents
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary

def generate_dictionary(documents):

    processed_docs = [doc.split() for doc in documents]
    
    # Create a Gensim dictionary
    dictionary = Dictionary(processed_docs)
    
    return dictionary, processed_docs

dictionary, processed_docs = generate_dictionary(sentences)

In [ ]:
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary

def generate_topic_words(topics, top_n_words):
    
    topic_words = []
    
    for topic_num, words in topics.items():
        if topic_num == -1:  # Skip the outlier topic
            continue
        # Collect only the top N words for each topic
        top_words = [word for word, _ in words[:top_n_words]]
        topic_words.append(top_words)
    
    return topic_words

def calculate_coherence_score(topics, dictionary, processed_docs, coherence='c_v', top_n_words=10):

    topic_words = generate_topic_words(topics, top_n_words)

    coherence_model = CoherenceModel(
        topics=topic_words,
        texts=processed_docs,
        dictionary=dictionary,
        coherence=coherence
    )
    
    # Step 4: Get the coherence score
    coherence_score = coherence_model.get_coherence()
    
    return coherence_score


## 3. Preparation for Model Evaluation

### 3.1 Import the annotated sample set

In [ ]:
annotated_df = pd.read_excel("/workspace/mijnidbcoachnlp/data/analysis_datasets/sample_sentences_for_labeling_100.xlsx", index_col=0)

In [ ]:
annotated_df #N = 328 sentences

In [ ]:
# print the unique labels
unique_labels = set(annotated_df["Label_Specific_Jiaxu"].to_list())
unique_labels

In [ ]:
len(unique_labels)
# there are 34 unique labels of this annotated sample set

### 3.2 Function to build model with different clustering methods

In [1]:
# set the vectorizer model
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.cluster import AgglomerativeClustering
from sklearn.cluster import KMeans
from hdbscan import HDBSCAN
from bertopic.vectorizers import ClassTfidfTransformer
from bertopic.representation import KeyBERTInspired



# configure the sub-models
#vectorizer
vectorizer_model=CountVectorizer(stop_words=dutch_stopwords, min_df=2, ngram_range=(1, 2), token_pattern=r'\b[a-zA-Z]{2,}\b')
# clustering (3 options)
hdb_model = HDBSCAN(min_cluster_size=20, min_samples=10, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
kmeans_model = KMeans(n_clusters=100)
agglo_model = AgglomerativeClustering(n_clusters=100)
# dimension reduction
umap_model=UMAP(n_neighbors=10, n_components=2, metric='cosine', random_state=42)
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)
representation_model = KeyBERTInspired()


NameError: name 'dutch_stopwords' is not defined

In [ ]:
# function to build the model with different selection of clustering method
# build the BERTopic model
from bertopic import BERTopic
from umap import UMAP
from bertopic import BERTopic

def build_bertopic(clustering_method):
    cluster_model=hdb_model
    if clustering_method == "KMeans":
        cluster_model = KMeans(n_clusters=50)
    
    if clustering_method == "Agglomerative":
        cluster_model = AgglomerativeClustering(n_clusters=50)
    

    # Initialize BERTopic with UMAP and HDBSCAN models
    topic_model = BERTopic(
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=cluster_model,
        vectorizer_model=vectorizer_model,
        top_n_words=10,
        verbose=True
    )

    # Fit-transform to get document topics and probabilities
    topics, probs = topic_model.fit_transform(sentences, embeddings)
    

    # Return only essential results
    return topic_model

## 4. BERTje-HDBSCAN model

In [ ]:
topic_model = build_bertopic(clustering_method = "HDBSCAN")

In [ ]:
topic_model.get_topic_info() #this is 20, 10 of HDBSCAN

In [ ]:
# Reduce outliers
new_topics = topic_model.reduce_outliers(sentences, topics)

In [ ]:
topic_model.update_topics(sentences, topics=new_topics)

In [ ]:
topic_model.get_topic_info()

In [ ]:
# hierarchical clustering based on HDBSCAN results
hierarchical_topics = topic_model.hierarchical_topics(sentences)

In [ ]:
topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

In [ ]:
topics_to_merge = [[194, 123, 28, 24, 213], # GI symptoms
                   [3, 4]] # health update
topic_model.merge_topics(docs, topics_to_merge)

In [ ]:
doc_info = topic_model.get_document_info(sentences)
doc_info[doc_info["Topic"] == 70]